In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집 II </b></font>

# 1절. selenium 을 이용한 동적 웹크롤링 문법
- https://selenium-python.readthedocs.io/
- `pip install selenium` (아나콘다 프롬프트)
    - 경고 무시 => pip install --upgrade requests (requests를 최신 버전으로 upgrade) 하거나, conda install urllib3==1.26.18
- selenium 버전 : 4.47 / requests 버전 : 2.28.1 / urllib3버전 : 2.7.0

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time 

In [3]:
dv = webdriver.Chrome()
dv.get('http://python.org')
elem = dv.find_element(By.NAME, 'q')
# By.CLASS_NAME, BY.ID, BY.CSS_SELECTOR, By.TAG_NAME
# a 태그에서 By.LINK_TEXT, By.PARTIAL_LINK_TEXT
elem.clear()
elem.send_keys('pycon')
elem.send_keys(Keys.RETURN) # enter

In [4]:
dv = webdriver.Chrome()
dv.get('http://python.org')
elem = dv.find_element(By.NAME, 'q')
elem.send_keys(Keys.CONTROL, 'a') # ctrl+a
elem.send_keys('pycon')
btn_elem = dv.find_element(By.CSS_SELECTOR, 'button#submit') #Go버튼
btn_elem.click()

In [5]:
result_list = dv.find_elements(By.CSS_SELECTOR, 'li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, result.get_attribute('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [6]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, result.attrs.get('href')))

PSF PyCon Trademark Usage Policy - /psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - /events/python-events/378/
PyCon Australia 2013 - /events/python-events/57/


In [7]:
from urllib.parse import urlparse
# https://www.python.org/search/?q=pycon&submit=
current_url = dv.current_url
print('현재 url :', current_url)
result_parse = urlparse(current_url)
print('url parsing 결과 :', result_parse)
domain = f'{result_parse.scheme}://{result_parse.netloc}'
domain = "{}://{}".format(result_parse.scheme, result_parse.netloc)
print('현재 domain :', domain)

현재 url : https://www.python.org/search/?q=pycon&submit=
url parsing 결과 : ParseResult(scheme='https', netloc='www.python.org', path='/search/', params='', query='q=pycon&submit=', fragment='')
현재 domain : https://www.python.org


In [8]:
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
# len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, domain+result.attrs.get('href')))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [9]:
dv.close() # 브라우저 종료

# 2절. 동적웹크롤링 예제
## 2.1 다음 뉴스 검색

In [11]:
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 lsit
driver = webdriver.Chrome()
url = 'http://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기

# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

검색할 단어는?치와와


In [12]:
bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
# len(bodies)
for body in bodies: 
    a = body.find_element(By.TAG_NAME, 'a')
    title = a.text
    link  = a.get_attribute('href')
    # print(title, link)
    news_list.append([title, link])

In [13]:
page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
# page_nav.text
nex_page = page_nav.find_element(By.LINK_TEXT, "4") # a태그의 text가 2인 a태그
nex_page.click()

In [14]:
import pandas as pd
pd.DataFrame(news_list, columns=['뉴스제목','링크']).shape

(10, 2)

## 2.2 다음 뉴스 페이징
- 위의 예제를 이용하여 원하는 페이지만큼 뉴스 검색 결과를 받아오기

In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time 
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = 'https://www.daum.net/'
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 뜰 때까지 0.5초 대기

# query = 'AI'
query = input('검색할 단어는?')
driver.find_element(By.CLASS_NAME, 'tf_keyword').send_keys(query)
driver.find_element(By.CSS_SELECTOR, 'button[type=submit]').click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기
# 뉴스 탭 클릭
# driver.find_elements(By.CSS_SELECTOR, 'ul.list_tab > li')[1].click()
driver.find_element(By.LINK_TEXT, '뉴스').click()

pages = int(input('몇 페이지 크롤링 할까요?'))
for page in range(1, pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR, 'strong.tit-g.clamp-g')
    for body in bodies:
        a = body.find_element(By.TAG_NAME, 'a')
        title = a.text
        link  = a.get_attribute('href')
        # print(title, link)
        news_list.append([title, link])
    page_nav = driver.find_element(By.CLASS_NAME, 'inner_paging')
    nex_page = page_nav.find_element(By.LINK_TEXT, str(page+1)) # a태그의 text가 2인 a태그
    nex_page.click()
    time.sleep(2)
driver.close()
news_df = pd.DataFrame(news_list, columns=['title','link'])
display(news_df.head())
print(news_df.shape)

검색할 단어는?말티즈
몇 페이지 크롤링 할까요?3


,title,link
0,"""나보다 목 길다""…SNS 달군 '가짜 말티즈' 두부",http://v.daum.net/v/20260517100148672
1,"'말티즈 X 먹지 않나요'…김재원 결국 사과했다 ""전국의 말티즈 견주께 죄송""[이슈S]",http://v.daum.net/v/20260415131123119
2,"“月7000원에 병원비 340만원”…카카오페이손보 펫보험, 4개월 만에 계약 1만 ...",http://v.daum.net/v/20260730135738684
3,"[현장포토] ""남자도, 청순해""...여상, 이스탄불 왕자님 (ISTfestival)",http://v.daum.net/v/20260815113323033
4,"""말티즈 똥 먹지 않나요""…김재원 공개사과 하게 한 강아지 '식분증'",http://v.daum.net/v/20260415171634121


(30, 2)


## 2-3 맞춤법 검사기
- 네이버 맞춤법 검사기 이용

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time 

In [32]:
driver = webdriver.Chrome()
driver.get('http://naver.com')
time.sleep(1)
elem = driver.find_element(By.ID, 'query')
elem.send_keys(Keys.CONTROL, 'a') # input이나 textarea나 다른 태그
elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(1)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
textarea.clear() # input이나 textarea
textarea.send_keys('안뇽하세요. 방갑습니다. 맛있는 점심시간 되세용')
btn = driver.find_element(By.CLASS_NAME, 'btn_check')
btn.click()
time.sleep(2)
result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
print(result)
# driver.close()

안녕하세요. 반갑습니다. 맛있는 점심시간 되세요


### 맞춤법검사_전.txt 파일을 맞춤법검사_후.tx로 파일 출력

In [126]:
# fp = open('data/ch14_맞춤법검사_전.txt', 'r', encoding='utf-8')
# text = fp.read()
# fp.close()
with open('data/ch14_맞춤법검사_전.txt', 'r', encoding='UTF8') as fp:
    text = fp.read()
ready_text_list = [] # 300자 기준으로 문장단위로 나눠진 text list
while len(text)>=300:
    temp = text[:300]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(read_text) for read_text in ready_text_list])

[282, 249, 198]


In [127]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time 
driver = webdriver.Chrome()
driver.get('https://www.naver.com')
time.sleep(0.5)
elem = driver.find_element(By.ID, 'query')

elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(0.5)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
results = '' # 맞춤법 검사 완료된 text
for idx, ready_text in enumerate(ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear() # input이나 textarea
    textarea.send_keys(ready_text)
    btn = driver.find_element(By.CLASS_NAME, 'btn_check')
    btn.click()
    time.sleep(1)
    # result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    result = soup.select_one('p._result_text.stand_txt').text
    results += result
    results = results.replace('.', '. ')

driver.close()

검사중...1/3
검사중...2/3
검사중...3/3


In [81]:
# 맞춤법 검사 결과(results)를 파일 출력
with open('data/ch14_맞춤법검사_후.txt', 'w') as fp:
    fp.write(results)

# 3절. 연습문제
- https://papago.naver.com/ 을 통해서 "data/ch14_맞춤법_후.txt"파일의 내용을 영문으로 번역하여 파일 출력

In [58]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time 

In [122]:
with open('data/ch14_맞춤법검사_전.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()
ready_text_list = [] # 300자 기준으로 문장단위로 나눠진 text list
while len(text)>=300:
    temp = text[:300]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(read_text) for read_text in ready_text_list])

[282, 249, 198]


In [129]:
driver = webdriver.Chrome()
driver.get('https://www.naver.com')
time.sleep(0.5)
elem = driver.find_element(By.ID, 'query')

elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(0.5)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
results = '' # 맞춤법 검사 완료된 text
for idx, ready_text in enumerate(ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear() # input이나 textarea
    textarea.send_keys(ready_text)
    btn = driver.find_element(By.CLASS_NAME, 'btn_check')
    btn.click()
    time.sleep(1)
    # result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    result = soup.select_one('p._result_text.stand_txt').text
    results += result
    results = results.replace('.', '. ')
    
driver.close()

검사중...1/3
검사중...2/3
검사중...3/3


In [130]:
with open('data/ch14_맞춤법후.txt', 'w') as fp:
    fp.write(results)

In [131]:
results

'안녕하세요.    AI로 작성한 문서는 아래와 같습니다.   인공지능(AI)은 현대 사회의 산업과 일상을 근본적으로 바꾸고 있는 혁신적 기술입니다.    방대한 데이터를 신속하게 처리하고 학습하는 능력 덕분에 수많은 분야에서 유용하게 활용되지만, 동시에 선결해야 할 과제도 함께 안고 있습니다.   주요 장점업무 효율성 및 생산성 극대화: 데이터 분석, 문서 요약, 단순 반복 업무를 자동화하여 작업 시간을 대폭 단축합니다.   24/7 연중무휴 작동: 피로감 없이 항상 최적의 상태로 서비스나 작업을 유지할 수 있습니다.   정확도 향상 및 오류 감소: 정교한 알고리즘을 통해 의료 진단, 금융 예측 등에서 인간의 실수를 줄여줍니다.  새로운 산업 창출: 자율주행, 맞춤형 콘텐츠 추천, 신약 개발 등 미래 신산업 생태계를 확장합니다.  주요 단점일자리 대체 불안: 단순노동부터 전문직 영역까지 일자리를 대체하면서 고용 불안을 유발할 수 있습니다.  윤리적 문제 및 편향성: 학습 데이터에 존재하는 편견이 결과물에 반영되거나, 저작권 및 사생활 침해 문제가 발생합니다.  기술 의존도 심화: AI에 대한 지나친 의존은 인간의 비판적 사고력과 문제 해결 능력을 약화시킬 위험이 있습니다. 높은 구축 및 유지 비용: 고성능 서버 구축과 데이터 관리에 막대한 자원과 비용이 소모됩니다. AI는 생산성 혁신을 이끄는 강력한 도구이지만, 윤리적 기준과 제도적 안전장치가 병행될 때 비로소 사회에 긍정적인 가치를 극대화할 수 있습니다. '

In [136]:
with open('data/ch14_맞춤법후.txt', 'r') as fp:
    text = fp.read()
ready_text_list = [] 
while len(text)>=3000:
    temp = text[:3000]
    last_dot_index = temp.rfind('.')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(read_text) for read_text in ready_text_list])

driver = webdriver.Chrome()
driver.get('https://papago.naver.com/')
time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME, 'entry-popup-module-scss-module__UZIxta__close')
if btn:
    btn.click()
    
input_elem = driver.find_element(By.CLASS_NAME,
                                 'text-translator-module-scss-module__CYJRkW__text-editor')

results = ''
for i, ready_text in enumerate(ready_text_list):
    print(f'번역중 ... {i+1}/{len(ready_text_list)}')
    input_elem.send_keys(Keys.CONTROL, 'a')
    input_elem.send_keys(ready_text)
    time.sleep(2)
    result = driver.find_elements(By.CSS_SELECTOR,
                             'div[class^="text-editor-module-scss-module"]')[1].text
    results += result
# driver.close()

[743]
번역중 ... 1/1


In [137]:
results

'Hello. The document created by AI is as follows.   Artificial intelligence (AI) is an innovative technology that is fundamentally transforming industry and everyday life in modern society.    Thanks to its ability to process and learn from massive amounts of data quickly, it is being used effectively across many fields, but it also carries the challenge of tasks that must be addressed first.   Key Benefits: Maximizing operational efficiency and productivity: Automate data analysis, document summarization, and simple repetitive tasks to significantly reduce working time.   24/7 year-round operation: it can maintain service or operations in optimal condition at all times without fatigue.   Improved accuracy and reduced errors: sophisticated algorithms reduce human mistakes in applications such as medical diagnosis and financial forecasting.  Creating new industries: expanding the ecosystem of future emerging industries such as autonomous driving, personalized content recommendation, and

In [138]:
with open('data/ch14_자동화영어번역본.txt', 'w') as fp:
    fp.write(results)